In [6]:
import os
import json

import numpy as np
import tensorflow as tf
from keras.applications import MobileNetV2
from keras.applications.mobilenet_v2 import preprocess_input
from keras.layers import Input, GlobalAveragePooling2D
from keras.models import Model
from PIL import Image, ImageFilter
import urllib.request
from urllib.parse import quote

import warnings
warnings.filterwarnings("ignore")

# Task one

In [7]:
categories = [
    "cat", "dog",
    "sun", "smiley face", "moon",
    "cup", "bucket",
    "apple", "tree", "house",
    "car", "bicycle", "clock",
    "flower", "fish",
]

RAW_DIR = "data/raw"
URL = "https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/{}.npy"

os.makedirs(RAW_DIR, exist_ok=True)

In [8]:
def download_category(name):
    path = os.path.join(RAW_DIR, f"{name}.npy")
    if os.path.exists(path):
        print(f"already have {name}")
        return path
    print(f"downloading {name} ...")
    urllib.request.urlretrieve(URL.format(quote(name)), path)
    return path

for name in category + heldout:
    download_category(name)

already have cat
already have dog
already have sun
already have smiley face
already have moon
already have cup
already have bucket
already have apple
already have tree
already have house
already have car
already have bicycle
already have clock
already have flower
already have fish
already have cell phone
already have envelope


In [9]:
rng = np.random.default_rng(42)

def sample_category(name, n):
    arr = np.load(os.path.join(RAW_DIR, f"{name}.npy"), mmap_mode="r")
    idx = np.sort(rng.choice(len(arr), size=n, replace=False))
    return np.array(arr[idx]).reshape(-1, 28, 28).astype("uint8")

samples = {name: sample_category(name, 500) for name in categories}

In [10]:
gallery_images, gallery_labels = [], []
query_images, query_labels = [], []

for label, name in enumerate(categories):
    imgs = samples[name]
    gallery_images.append(imgs[:450])
    gallery_labels += [label] * 450
    query_images.append(imgs[450:])
    query_labels += [label] * 50

gallery_images = np.concatenate(gallery_images)
gallery_labels = np.array(gallery_labels)
query_images = np.concatenate(query_images)
query_labels = np.array(query_labels)

print("gallery :", gallery_images.shape)
print("queries :", query_images.shape)

gallery : (6750, 28, 28)
queries : (750, 28, 28)


# Task two

In [12]:
backbone = MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights="imagenet"
)

backbone.trainable = False

inputs = Input(shape=(160, 160, 3))
x = backbone(inputs)
outputs = GlobalAveragePooling2D()(x)

model = Model(inputs=inputs, outputs=outputs)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_160            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,257,984 (8.61 MB)

In [13]:
def to_model_input(images_28, invert=False):
    images = np.array(images_28, dtype="float32")
    if invert:
        images = 255.0 - images
    images = np.expand_dims(images, axis=-1)
    images = tf.image.resize(images, (160, 160)).numpy()
    images = np.repeat(images, 3, axis=-1)
    images = preprocess_input(images)
    return images

In [14]:
def embed(model, images_28, invert=False, batch_size=64):
    features = []
    for i in range(0, len(images_28), batch_size):
        batch = to_model_input(images_28[i:i + batch_size], invert=invert)
        features.append(model.predict(batch, verbose=0))
    features = np.concatenate(features, axis=0)

    features = features / np.linalg.norm(features, axis=1, keepdims=True)
    return features

In [15]:
embeddings = embed(model, gallery_images)
print("Embedding shape:", embeddings.shape)

os.makedirs("index", exist_ok=True)
np.save("index/embeddings.npy", embeddings)
np.save("index/gallery_images.npy", gallery_images)
np.save("index/gallery_labels.npy", gallery_labels)

Embedding shape: (6750, 1280)


# Task three

In [16]:
def to_gray(image):
    if hasattr(image, "mode") and image.mode not in ("RGB", "RGBA", "L"):
        image = image.convert("RGBA")
    img = np.asarray(image).astype("float32")

    if img.ndim == 2:
        return img

    if img.shape[2] == 4:
        alpha = img[..., 3:4] / 255.0
        img = img[..., :3] * alpha + 255.0 * (1.0 - alpha)
    else:
        img = img[..., :3]

    return img @ np.array([0.299, 0.587, 0.114], dtype="float32")

In [17]:
def get_strokes(gray, threshold=128):
    return ((gray < threshold) * 255).astype("uint8")

In [18]:
def crop_square(strokes, margin=0.1):
    ys, xs = np.where(strokes > 0)
    if len(ys) == 0:
        return None

    cropped = strokes[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    h, w = cropped.shape

    side = max(h, w)
    pad = int(side * margin)
    out = np.zeros((side + 2 * pad, side + 2 * pad), dtype=strokes.dtype)
    top = (out.shape[0] - h) // 2
    left = (out.shape[1] - w) // 2
    out[top:top + h, left:left + w] = cropped
    return out

In [19]:
def resize_28(squared):
    img = Image.fromarray(squared)
    out = np.array(img.resize((28, 28), Image.BOX))

    if out.max() < 230:
        size = max(3, (squared.shape[0] // 28) | 1)
        thick = img.filter(ImageFilter.MaxFilter(size))
        out = np.array(thick.resize((28, 28), Image.BOX))

    return (out.astype("float32") / out.max() * 255).astype("uint8")

In [20]:
def preprocess_query(image, threshold=128):
    gray = to_gray(image)
    strokes = get_strokes(gray, threshold)
    squared = crop_square(strokes)
    if squared is None:
        return None
    return resize_28(squared)

In [21]:
def search(query_vector, embeddings, k):
    query_vector = np.array(query_vector).flatten()
    scores = np.dot(embeddings, query_vector)
    sorted_indices = np.argsort(scores)[::-1]
    top_indices = sorted_indices[:k]
    top_scores = scores[top_indices]
    return top_indices, top_scores

In [22]:
query_embeddings = embed(model, query_images)
print("query embeddings:", query_embeddings.shape)

in_scores = []
for query in query_embeddings:
    indices, scores = search(query, embeddings, 1)
    in_scores.append(scores[0])
in_scores = np.array(in_scores)

query embeddings: (750, 1280)


In [23]:
threshold = np.percentile(in_scores, 5)

config = {
    "threshold": round(float(threshold), 4),
    "top_k": 25,
    "invert": False
}

with open("index/config.json", "w") as file:
    json.dump(config, file)

print("Search Configuration")
print(config)

Search Configuration
{'threshold': 0.8012, 'top_k': 25, 'invert': False}


## For app

In [24]:
import shutil
from google.colab import files

shutil.make_archive("index", "zip", "index")

files.download("index.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>